# Attachment sensitivity investigation workbook

**Status: COMPLETED 29 September 2026** (prepared 25 September 2026). All six arms were run locally at 0-6 ft, verified (30/30 checks), and analyzed with 199-success bootstraps under both specifications. Report: `outputs/attachment_sensitivity/attach_20260925/REPORT.md`; index: `outputs/attachment_sensitivity/attach_20260925/README.md`.

This is a separate workbook for investigation design, agent delegation, reproducible execution records, and later results. The complete agent prompt is embedded below and also saved in `docs/attachment_sensitivity/agent_prompt.md`.

Opening or running this notebook does not launch graph construction or cluster jobs. Read-only source-data audits are opt-in. Heavy computation belongs in tested, resumable scripts that the investigation agent will implement.

The purpose is to evaluate the assumptions, including unfavorable findings; it is not to validate a preselected robustness claim.

## 1. Proposed manuscript text

The VA TO DO paragraph has been replaced by a past-tense account of the completed tests (29 September 2026). Numeric findings belong in Results; see REPORT.md section 7 for suggested text.

```latex
%% Draft attachment paragraphs for Section 4.2.
%% The stress tests were completed on 2026-09-29 (experiment attach_20260925).
%% diestel2005 is a suggested bibliography key; reconcile it with cas-refs.bib.

Each block's representative point is attached to the nearest node in the
raw network's largest connected component, defined as the largest set of
nodes linked by road paths. We apply a maximum straight-line attachment
distance of 2~km and exclude origins exceeding this threshold. A larger
tolerance is allowed for block origins than for facilities because
representative points describe spatial areas and need not coincide with
road access locations, whereas facility coordinates identify specific
destinations. Attachments remain fixed across scenarios, so origins are
not reassigned to alternative roads after disruption.

Public and private schools and fire stations form a pooled set of facility
destinations. Facility attachment emphasizes redundancy in the surrounding
road network: attaching a facility to a dead-end driveway would make that
final segment a shared dependency of every route, regardless of alternatives
in surrounding streets. We preferentially attach facilities to the nearest
node within 1~km that lies in the raw network's largest connected component
and belongs to a two-edge-connected component containing at least two nodes.
Within such a component, each pair of distinct nodes is connected by at least
two paths that share no edges \citep{diestel2005}. If no qualifying node lies
within 1~km, we use the nearest node in the full raw network within the same
distance threshold; otherwise, the facility is excluded. Attachments remain
fixed across scenarios. This procedure abstracts from the immediate
connection between a facility and its assigned node, which may itself be
vulnerable to disruption.

No single attachment distance can be assumed to represent access equally
well across blocks and facilities with different spatial configurations.
We therefore treat the 2-km origin and 1-km facility limits as operational
tolerances whose influence requires evaluation.
We assessed the influence of these tolerances with five alternative
specifications, holding all other modeling choices fixed: origin limits of
1~km and 500~m; a 500-m facility limit applied to both the preferred and
fallback searches; admission of every previously excluded candidate facility
at its nearest network node without a distance limit, with all existing
attachments unchanged; and nearest-node facility attachment without the
preference for two-edge-connected components. For each specification we
reclassified all blocks at 0--6~ft, rebuilt the block-group transition data,
and re-estimated all seven transition models under both covariate
specifications (Section~\ref{sec:sensitivity}).

%% Completed 2026-09-29. Results text and every number: outputs/attachment_sensitivity/
%% attach_20260925/REPORT.md, sections 3-4 and 7.

%% AFTER the investigation: replace VA TO DO with a past-tense account of
%% the tests actually completed. Report failed/incomplete runs separately.
%% Put numeric findings in Results, not an assumed success claim in Methods.
%% If supported, a Results sentence could read:
%% "Across the tested attachment specifications, [metric] changed by
%% [absolute change and range], while [specified substantive findings]
%% remained consistent. These comparisons indicate limited sensitivity to
%% the tested attachment choices, although individual connections remain
%% approximations of actual facility access."
%% If outcomes differ materially, describe which rules, places, populations,
%% and associations change; revise the headline rather than claiming robustness.

```

## 2. Configuration and read-only inventory

Set `SLR_REPO` in the environment if launching outside the repository. Enable the existing-data audit only if desired; it does not rerun any sensitivity treatment.

In [ ]:
import os
import sys
import platform
from pathlib import Path

configured_root = os.environ.get("SLR_REPO")
if configured_root:
    REPO = Path(configured_root).expanduser().resolve()
else:
    candidates = []
    for ancestor in (Path.cwd(), *Path.cwd().parents):
        candidates.extend([ancestor, ancestor / "slr_fl_fragile_access"])
    REPO = next((p.resolve() for p in candidates
                 if (p / "scripts" / "02_access_flags.py").is_file()), None)
    if REPO is None:
        raise RuntimeError("Set SLR_REPO to the repository root before running this workbook.")
if not (REPO / "scripts" / "02_access_flags.py").is_file():
    raise RuntimeError(f"Invalid repository root: {REPO}")

RUN_READONLY_AUDITS = False
REFERENCE_RUN = REPO / "data/processed/access/edited/della_runs/positive_layer_20260814_approach"
print("Repository:", REPO)
print("Interpreter:", sys.executable)
print("Platform:", platform.platform())
for relative in ["run_manifest.json", "service_snapping_audit.csv", "block_access_flags_long.parquet"]:
    candidate = REFERENCE_RUN / relative
    print(relative, "present" if candidate.is_file() else "MISSING")
try:
    import psutil
    print("Available RAM (GiB):", round(psutil.virtual_memory().available / 2**30, 1))
except ImportError:
    print("RAM not inspected: psutil unavailable; no installation attempted.")


## 3. Prespecified run registry

All rows start as **not run**. The agent must record actual stage completion, manifests, commands, elapsed times, resource use, and local/Slurm identifiers. All treatments use the same candidate-facility footprint and the approach bridge rule.

In [ ]:
import pandas as pd

planned_runs = pd.DataFrame([
    ("reference", 2000, 1000, "preferred_then_fallback", "Reproduce current behavior"),
    ("origin_1000", 1000, 1000, "reference_facilities_fixed", "Tighter origin tolerance"),
    ("origin_500", 500, 1000, "reference_facilities_fixed", "Stronger origin restriction"),
    ("facility_500", 2000, 500, "preferred_then_fallback", "Tighter facility policy"),
    ("facility_add_uncapped", 2000, None, "preserve_valid_add_excluded_nearest", "Isolate facility exclusion"),
    ("facility_nearest_1000", 2000, 1000, "unconstrained_nearest", "Test component preference"),
], columns=["attachment_arm", "origin_limit_m", "facility_limit_m", "facility_rule", "purpose"])
planned_runs["status"] = "COMPLETED (local, attach_20260925)"
planned_runs["models"] = ""
planned_runs.loc[planned_runs.attachment_arm.eq("facility_add_uncapped"), "models"] = "reused from reference (identical input)"
planned_runs.loc[planned_runs.attachment_arm.ne("facility_add_uncapped"), "models"] = "fit, both specs, 199 reps"
planned_runs["bridge_rule"] = "approach"
planned_runs["scenarios"] = "0,1,2,3,4,5,6"
print(planned_runs.to_string(index=False))


## 4. Optional audit of existing attachments

This cell only summarizes the saved primary run. These are **coverage diagnostics, not sensitivity results**. A reduction in eligible origins can be counted without demonstrating that model findings remain stable. Facility candidates and retained facilities are different counts. Distance penalties are not node displacements.

In [ ]:
if not RUN_READONLY_AUDITS:
    print("Skipped: set RUN_READONLY_AUDITS=True for read-only source diagnostics.")
else:
    columns = ["slr_ft", "origin_snap_distance_m", "analysis_eligible", "land_area_m2", "pop20"]
    existing = pd.read_parquet(REFERENCE_RUN / "block_access_flags_long.parquet", columns=columns)
    baseline = existing.loc[existing.slr_ft.eq(0)].copy()
    eligible = baseline.loc[baseline.analysis_eligible.eq(True)].copy()
    print("Eligible origin distances (m):")
    print(eligible.origin_snap_distance_m.describe(percentiles=[.5, .9, .95, .99]))
    coverage = []
    for threshold_m in [500, 1000, 1500, 2000]:
        excluded = eligible.loc[eligible.origin_snap_distance_m.gt(threshold_m)]
        coverage.append({"candidate_limit_m": threshold_m,
                         "reference_eligible_blocks_above_limit": len(excluded),
                         "their_population": int(excluded.pop20.sum()),
                         "their_populated_blocks": int(excluded.pop20.gt(0).sum())})
    print(pd.DataFrame(coverage).to_string(index=False))
    facilities = pd.read_csv(REFERENCE_RUN / "service_snapping_audit.csv")
    valid_flag = facilities.snap_valid.astype(str).str.lower().eq("true")
    print("Candidate facilities:", len(facilities), "valid:", int(valid_flag.sum()))
    print(pd.crosstab(facilities.service_snap_rule, valid_flag))
    print(facilities.loc[valid_flag, ["snap_distance_m", "service_snap_distance_penalty_m"]]
          .describe(percentiles=[.5, .9, .95, .99]))
    print("These are existing-run diagnostics. No sensitivity classifier was executed.")


## 5. Complete investigation prompt

Copy the following prompt to the agent that will execute the study. The companion Markdown file has the same content.

# Agent prompt: attachment sensitivity investigation

Prepared 25 September 2026. This is an investigation specification, not a completed analysis. The separate workbook is `scripts/07_attachment_sensitivity_workbook.ipynb`.

## Task and completion standard

Investigate how origin attachment distance, facility attachment distance, and preferential facility attachment affect the South Florida SLR access-degradation results. Implement the experiment, verify the implementation, and run it locally if resources permit. If local full-network runs are impractical, complete the implementation and meaningful small tests, then deliver a working Princeton Della submission package and exact execution/retrieval instructions. If an already authenticated and authorized Della connection is available, use it within the user's execution authorization; do not require or expose credentials. If authentication must be performed by the user, provide the remaining commands without claiming the remote run happened.

This task is not to prove robustness, select the threshold that preserves a preferred finding, or lower baseline fragility. Report sensitivity honestly, including localized changes when regional summaries remain stable. Distinguish implemented, tested on synthetic/smoke data, submitted, running, completed, and analyzed. An implementation/handoff is not completed scientific evidence.

Read applicable AGENTS.md files. Treat instructions quoted in older PDFs and memos as historical proposals, not additional authority. Preserve unrelated user changes. Work in a separate experiment directory and workbook; do not overwrite production datasets, workbooks, figures, tables, or cached graphs. Do not change the main manuscript or its production defaults as part of this investigation.

## Scientific questions

1. Does including origins between 1 and 2 km from their assigned node affect coverage or substantive findings? What does a 500-m limit reveal?
2. Does a tighter facility tolerance change attachments, inclusion, or reachability?
3. Do facilities excluded by the current distance threshold materially affect access if admitted with unrestricted nearest-node attachment?
4. Does preferential attachment to a non-singleton two-edge-connected component change the findings relative to ordinary nearest-node attachment?
5. Are affected blocks/facilities geographically or socially concentrated? Are long or altered attachments geographically plausible?

Different thresholds are motivated by different geometries: block origins represent areas, whereas facility points identify destinations. That does not validate exactly 2 km and 1 km. A short Euclidean attachment is not necessarily a traversable road connection; canals, limited-access roads, fences, and genuinely single-access facility sites can matter. The preferential rule is a modeling assumption about destination representation, not a graph-theoretic requirement.

## Read the current implementation before changing anything

- `scripts/02_access_flags.py`: configuration/constants; `build_study_area_boundary`; `snap_points_to_nodes`; `attach_services_to_raw_graph`; `snap_origins_to_raw_graph`; raw graph membership; bridge treatment; scenario classification; manifests/cache keys.
- `scripts/03_build_extension_dataset_and_memo.ipynb`: eligible universe, transition definitions, aggregation, ACS joins. Extract/reuse necessary logic without rerunning all plotting or overwriting existing outputs.
- `scripts/04_shared_model_spec.R` and `scripts/04_transition_models.R`: seven grouped-binomial models, risk denominators, standardization, AMEs/bootstrap, alternate physical specification.
- `scripts/05_population_figures.py`: five baseline-connected transitions and population totals.
- `scripts/02d_measurement_validity_diagnostics.py`, `scripts/02g_compare_bridge_rule_runs.py`: reusable audits, with careful attention to hardcoded paths/rule labels.
- `docs/methods_outline_20260925.tex`, `docs/physical_covariate_and_conley_extension.md`, `reports/03_arm_robustness.md`, and `outputs/run_comparison/final_methods_verification.md` for context. Reports can be stale: inspect live output workbooks and manifests.
- Canonical reference access run: `data/processed/access/edited/della_runs/positive_layer_20260814_approach/` (manifest, service audit, access Parquet).
- Primary model tables: `outputs/tables/ame_bootstrap_results_approach.xlsx` and `_approach_with_physical.xlsx`; primary input datasets under `data/processed/analysis/`.

As of preparation, `slurm/run_access_flags.sbatch` referenced by the workflow README does not exist. The supplied `docs/attachment_sensitivity/della_job.sbatch.template` is a generic guarded launcher, not an implemented sensitivity runner. Produce the actual runnable experiment and Slurm files; do not merely repeat nonexistent commands from the README.

## Current behavior to preserve and verify

- Origins: polygon representative points, nearest raw-largest-connected-component node, maximum 2,000 m; attachments fixed across scenarios. Positive land area plus valid origin snap determines eligibility. Do not add a population-positive filter.
- Candidate facilities: the same school/fire-station records within 10 km of the **retained-network bounding rectangle**, not the county polygon. Preserve candidate IDs, geometry, multiplicity, and facility types across arms. Do not silently deduplicate differently.
- Primary facilities: nearest raw-LCC node in a two-edge-connected component with at least two nodes within 1,000 m; fallback to the nearest node of the full raw graph within 1,000 m; otherwise exclude. This is a geometric nearest-node search, not tracing the actual driveway to its street entrance.
- All runs: same retained road PBF and classes; private-access exclusion; graph coordinate construction; representative points; positive layer gate; `approach` bridge rule; same NOAA 0--6 ft extents; fixed geographic extent; same facility pooling and facility-operability assumptions.
- Apply inundation and bridge rules at 0 ft too. Do not treat the raw graph as the baseline dry graph.
- Do not activate `--legacy-mode`. `--legacy-service-snap` may implement one narrow treatment, but verify precisely which behavior it changes before using it.
- Keep invalid/nonfinite facility geometry invalid even in an uncapped arm. No artificial nodes, new roads, or artificial access links are added to the graph.

## Required run matrix

Six arms, each evaluated at 0, 1, 2, 3, 4, 5, and 6 ft:

| ID | Origin limit | Facility treatment | Purpose |
|---|---:|---|---|
| `reference` | 2,000 m | Existing preferential rule, 1,000 m including fallback | Reproduce current behavior |
| `origin_1000` | 1,000 m | Exactly the reference facility map | Test the larger origin tolerance |
| `origin_500` | 500 m | Exactly the reference facility map | Stronger origin restriction |
| `facility_500` | 2,000 m | Existing preferential-plus-fallback algorithm, both capped at 500 m | Tighter facility attachment policy |
| `facility_add_uncapped` | 2,000 m | Preserve every reference-valid attachment; add reference-excluded candidates at their unconstrained nearest raw-graph node, with no finite distance cap | Isolate exclusion of distant facilities |
| `facility_nearest_1000` | 2,000 m | Attach every candidate to its unconstrained nearest full-raw-graph node within 1,000 m; no component preference | Test preferential attachment separately |

`facility_add_uncapped` is NOT permission to increase the preferred-node radius for already included facilities, move all destinations, expand the candidate geography, or download statewide extra facilities. An uncapped preferred-node search would be a different experiment and is outside the required matrix. Store the distance-limit state explicitly (e.g. JSON null plus a named rule), rather than emitting nonstandard JSON Infinity.

Under `facility_500`, an existing preferred snap beyond 500 m may be replaced by a valid unconstrained snap within 500 m; implement the actual algorithm, not just deletion of all current snaps over 500 m. Report additions, exclusions, and reattachments separately. Ordinary nearest-node treatment can also change facility eligibility; distinguish this from pure relocation in reporting.

## Isolation, provenance, and implementation architecture

1. Record git state/source hashes, input hashes, software versions, resolved paths, and existing output hashes before any mutation. Existing Della manifests record a dirty source tree: do not invent byte-for-byte provenance from a commit alone.
2. Put experiment outputs below `outputs/attachment_sensitivity/<experiment_id>/` and new intermediate access data below `data/processed/access/attachment_sensitivity/<experiment_id>/`. Never name a facility treatment as if it were a new bridge arm. Bridge rule stays `approach`; use a distinct `attachment_arm` field.
3. Build a small command-line runner/config interface with explicit treatment, thresholds, input/output/cache paths, scenarios, and reference attachment map. Reuse production functions where safe, preserving defaults. If editing shared functions is necessary, add default-equivalence tests and keep original CLI behavior unchanged.
4. IMPORTANT: `04_transition_models.R --data ...` changes the input but still writes arm-tagged outputs to `outputs/tables/`. Merely passing a different dataset would overwrite primary results. Add safe output-directory/run-tag support or use a verified isolated experiment entry point. Inspect every downstream writer, including notebook exports and population figures, for the same issue.
5. Use stable facility identifiers and retain a reference attachment registry. The production audit includes rejected candidates, not only retained facilities. Separate candidate counts, valid attachments, and in-graph reachable facilities in every scenario.
6. Write run/configuration manifests, stage-completion markers after validation, timings, and resume behavior. Do not treat partially written outputs as complete; writes should be atomic where practical. Cache keys must distinguish anything that affects a cached artifact.

## Efficient execution without changing the experiment

The raw road topology is unchanged across these arms. Verified segmentation and raw-membership caches can be reused. Reconstruct the graph from compatible caches if appropriate; do not rebuild identical geometry six times just to satisfy a phrase in the task. Build/load each scenario's retained graph once per suitable worker, then recompute facility-dependent connected-component/service counts and access classifications for each distinct facility map.

Origin-only thresholds change valid-origin membership, not facility placement or road topology. If code inspection and equivalence tests establish this, their eligible results can be derived from the full reference block records, with correct eligibility/exclusion flags and complete downstream rebuilding. Do not drop rows blindly or claim derived output is a fresh full classifier run. Document the exact reuse and prove shared retained blocks' states match. Facility treatments require new facility-dependent classification unless full relevant equivalence is proved.

Avoid multiple jobs writing one cache simultaneously. Either warm an isolated cache once and use it read-only, or use per-job caches. Avoid `--smoke` for scientific conclusions because it clips the network and changes connectivity; use it only to exercise code paths. A small full-network origin sample is useful for timing but still requires graph memory.

## Verification required before expensive runs

Use small synthetic networks with meaningful cases, not only tests mirroring the code:

- A loop/grid with a dead-end facility spur: nearest and preferred attachment differ for the intended reason.
- A real single-access facility site: document how the preferred rule bypasses that dependency instead of claiming it is universally a mapping error.
- A preferred node outside 500 m but an ordinary node inside 500 m: verify the fallback in `facility_500`.
- A reference-excluded facility with a finite distant nearest node: only the add-uncapped arm admits it, preserving reference-valid node assignments exactly.
- An invalid-coordinate candidate remains excluded when the distance cap is removed.
- Origins with distances on either side of and exactly equal to each threshold: confirm the inclusive <= rule, land-area eligibility, and zero-population handling.
- Two geographic road networks separated by a barrier: expose Euclidean attachment limitations; do not add an unrequested barrier-routing correction.

Reproduce the reference with current source and explain discrepancies before attributing any difference to sensitivity. Required checks include eligible GEOID sets; population totals; facility node assignments; bridge removals; inundation flags; four access-state flags; baseline transitions; and seven-model samples/point estimates. Existing reference totals are 70,695 input blocks, 68,521 eligible blocks, 6,135,688 eligible residents, and 2,163 retained facilities; verify against the actual files. Reproducing aggregate counts alone is insufficient.

For facility-only arms, origin membership and origin inundation flags must be identical. For add-uncapped, reference-valid attachments must match exactly and destinations form a superset; reachability/redundancy at a given water level should not worsen solely from adding facilities. Baseline-relative transition totals need not be monotone because the baseline itself can improve. Investigate any classification invariant violation, including coincident origin/facility nodes, rather than invoking a mathematical theorem to waive a code discrepancy. Unexpected classifier defects are findings: do not silently repair them only in an experimental arm.

## Attachment audit and map review

Export distributions and individual records for origin distance, facility distance, fallback use, candidate exclusion, preferred-node changes, old/new node IDs, and facility component membership. Include median, p90, p95, p99, maximum, affected block counts, and affected population. Separate all blocks from populated blocks for description without changing eligibility.

Important: added snap distance (new facility-to-node distance minus old facility-to-node distance) is NOT the physical displacement between old and new nodes. Compute both. A small difference in the two distances can conceal a move across a canal. Avoid repeating the old memo's interpretation that a median distance penalty proves all changes are tiny or correct.

Produce an inspectable GeoPackage/HTML map or map panels for changed/distant attachments. Include a deterministic sample from short and long distance bands, changed fallback cases, relevant counties/facility types, and the most influential changed areas. Inspect actual imagery/road connections only when suitable data are accessible; otherwise label geographic plausibility unverified. Do not manufacture an audited entrance or claim manual validation based on distances alone.

## Outcomes, denominators, and model comparisons

Produce, by arm and water level:

1. Candidate/retained/reattached/fallback facility counts, origin eligibility counts and population, exclusion reasons, and sample-composition summaries.
2. Baseline and scenario state counts/populations/shares and full reference-vs-arm state cross-tabs by GEOID. Report both absolute changes and changes in percentage points with explicit denominators.
3. The five baseline-connected transitions: redundant to fragile/isolated/inundated; fragile to isolated/inundated. Each arm uses its OWN baseline states. Define composite worse outcomes consistently.
4. Population totals for inundation only, isolation-or-inundation, and fragility-or-worse, and the non-inundation pathway share, separately at each scenario. Follow `05_population_figures.py`: these population summaries concern baseline-connected origins. Do not mix them with broader diagnostic new-inundated flags that include baseline-isolated origins. Do not sum people across scenarios as unique residents.
5. Geographic and demographic summaries of changed/excluded origins; use descriptive comparisons without pretending repeated blocks create independent evidence about ACS attributes.

Report two comparison views: (a) each arm's operational eligible sample and (b) pairwise common eligible origins. Origin restrictions should not change retained common origins' classifications; resulting model/sample differences reflect coverage. For facility arms, compare changes in baseline risk membership as well as subsequent outcomes. Any common-baseline-state regression check must explicitly condition on the intersection of baseline-risk origins; do not silently replace the primary arm-specific risk sets.

Rebuild block-group transition datasets; reuse the same ACS values and physical covariates by GEOID instead of making new API pulls. Reproduce the six social covariates, income transform, standardization order, complete cases, county/scenario fixed effects, and baseline block-count denominators. Zero-population eligible blocks remain included. Main AMEs are equally averaged over model rows, not population weighted. Do not confuse this experiment with the existing population-weighted-AME sensitivity.

For every arm with changed model inputs, fit all seven models under demographic-only AND physical-adjusted specifications. The age finding depends on specification, so do not validate only racial-composition coefficients. Use at least the current 199 successful block-group bootstrap replications for final comparable tables, the documented seeds/retry policy, percentile intervals, and the current normal-approximation p-values based on bootstrap SDs. Short bootstrap runs are smoke tests only. Report realized successful/failed replications and sample counts. Use consistent scaling within clearly labeled common-support comparisons. Where an entire model input and specification is exactly identical, reuse its verified result and document the equality instead of refitting needlessly.

Compare AME signs, magnitudes in probability and percentage-point units, intervals, and sample changes for ALL six social terms. Relative change is unstable near zero; report absolute differences too. A p=0.049 to p=0.051 crossing alone is not evidence of a materially different effect, and overlapping intervals are not a formal equivalence test. Do not claim significant differences between specifications without an appropriate comparison procedure. Sensitivity conclusions concern the tested choices, not proof that the network depicts every real connection correctly.

Before looking at sensitivity outcomes, record which manuscript conclusions and metrics will be assessed (baseline fragility, added population through non-inundation pathways, and social associations under both specs). Report the full ranges. Do not invent a universal pass/fail tolerance or stop checking once selected findings look stable.

## Local execution versus Della

Inspect available RAM, disk space, existing dependencies, compatible caches, and prior timing/resource logs before a full local run. The local environment previously used is `C:/Users/Vivek/miniforge3/envs/research-geo/python.exe`; verify it rather than using the Windows Store python alias. Use explicit interpreters, environment isolation, captured logs, and bounded worker/thread counts. The graph is largely serial NetworkX work; extra allocated CPUs do not automatically accelerate it. Avoid memory overcommit from concurrent graphs. Do not launch an uncontrolled full run to discover whether it fits.

Run locally if preflight and a representative benchmark demonstrate feasibility. If not, finish smoke/equivalence tests locally and prepare a complete Della handoff. Do not keep trying full local runs after reproducible resource failures. A Della benchmark should use a compute allocation and `jobstats`/`sacct` to size the full request. Do not infer peak full-network memory from a clipped smoke graph.

Use current official guidance and verify it again at execution time:

- Della and scheduler policies: https://researchcomputing.princeton.edu/systems/della
- Python environments: https://researchcomputing.princeton.edu/support/knowledge-base/python
- Memory requests: https://researchcomputing.princeton.edu/support/knowledge-base/memory
- First Slurm job: https://researchcomputing.princeton.edu/get-started/guide-princeton-clusters/3-first-slurm-job
- Job profiling: https://researchcomputing.princeton.edu/support/knowledge-base/jobstats

At preparation, Della uses Slurm; full graph/model computations belong on compute nodes. CPU work does not require a GPU. Let the scheduler select QOS unless current documentation requires otherwise; do not invent an account or partition. Discover available modules and reuse a working project environment or create a pinned isolated environment. The project manifest is useful for versions but does not prove a given environment exists on Della.

Historical manifests show `/scratch/gpfs/ERICTATE/vivek/slr_fl_fragile_access`; treat it as a path to verify, not a guaranteed account, current checkout, or writable allocation. Scratch is not a durable backup. Prefer existing cluster inputs with matching checksums to retransferring large files. A fresh git checkout will not contain the ignored `data/` directory. Provide an explicit input inventory and exact staging/symlink plan; do not rewrite canonical paths to absent data or use commands that overwrite unrelated files.

Required Della package:

1. A real, tested experiment CLI and validated configuration manifest. No command may depend on a flag that was merely proposed in this prompt.
2. Pinned environment requirements and an environment-setup script with verified module/env names, or clear user-specific fields if remote discovery is unavailable.
3. An entry-point shell script invoking the actual implemented stages, with failure propagation and logs.
4. An sbatch file (one node, explicit tasks/CPUs/memory/time, log locations created before submission). Explain measured or provisional resource sizing. Supply a small compute-node benchmark first if no full-network profile exists.
5. Separate graph and model stages if this helps resources/restarts, with `afterok` dependencies or a safe sequential pipeline. Use isolated/read-only caches. Serial arm execution or a limited array is preferable to uncontrolled simultaneous graph copies.
6. Copy-paste commands for connecting/staging, preflight, environment preparation, submission (`sbatch --parsable`), monitoring (`squeue`, logs), resource inspection (`sacct`/`jobstats`), cancellation of this investigation's job IDs only, resumption, and fetching the small tables/maps/manifests needed locally.
7. An explicit note that a submitted/pending job is not a completed result and that an afterok postprocessing job will not execute after a failed dependency without resubmission.

The supplied generic template accepts a project environment-setup file and an implemented entry-point file via environment variables. Turn it into the concrete scripts above. If remote execution is unavailable, do not claim to have verified paths, modules, resource sufficiency, or sbatch acceptance. Clearly separate tested local code from user-side cluster steps.

## Deliverables

- Update `scripts/07_attachment_sensitivity_workbook.ipynb` with provenance, run registry, commands/job IDs, validation results, tables, maps, and interpretation. Keep heavy runs in resumable scripts rather than a notebook cell that runs automatically. It must remain safe to open without starting jobs.
- An experiment run matrix/config file and isolated implementation with focused tests.
- `attachment_audit.csv`, `eligibility_comparison.csv`, `state_comparison.csv`, `transition_population_comparison.csv`, model-input/sample diagnostics, complete AME comparison tables for both specifications, and a spatial review artifact. Exact filenames may differ if clearly indexed in a README.
- A completed-results report with quantified changes and limitations, or a candid partial report plus complete Della runbook if runs cannot be completed in-session. Never fill result placeholders with hoped-for robustness.
- The actual local command or Della entry point/sbatch/environment files and step-by-step instructions.
- Suggested final Methods and Results text. Replace VA TO DO only for completed analyses and support every numerical sentence with an output path/table field. If thresholds or the preferred snapping assumption materially affect findings, say so and identify what needs revision.

## Known baseline observations, not sensitivity findings

A read-only audit of the saved primary baseline on 25 September 2026 found eligible-origin snap distances: median 47.827 m; p95 196.682 m; p99 535.754 m. There are 196 eligible blocks (7,251 residents) above 1,000 m and 777 (52,758 residents) above 500 m. These are coverage diagnostics, not rerun results or validated thresholds. Reproduce them from the saved Parquet.

The older facility memo's 13.5-m statistic is an added distance penalty, not measured node displacement. Some older bridge reports describe 49 bootstrap replications, whereas current principal workbooks contain 199. Current physical-adjusted older-adult intervals include zero for all seven models; avoid copying older unqualified claims about that association.

Start by inventorying the existing run and producing the isolated implementation plan, then execute the authorized work without repeatedly asking whether to continue. Request only genuinely missing credentials, account/path choices, or an interpretation decision that cannot be resolved from the repo. Do not select or suppress results to obtain the hoped-for conclusion.


## 6. Execution handoff

# Attachment sensitivity workbook and handoff

Status: **planned; sensitivity runs have not been performed**. Existing production results were only inspected while preparing this package.

## Files

- `../../scripts/07_attachment_sensitivity_workbook.ipynb`: separate Jupyter workbook with the complete agent prompt, six-arm registry, read-only audit cells, manuscript draft, and execution handoff.
- `agent_prompt.md`: the same detailed prompt in a form that is easy to copy to an agent.
- `manuscript_draft.tex`: replacement attachment paragraphs and a visible **VA TO DO** stress-test paragraph. No favorable outcome is presumed.
- `della_job.sbatch.template`: generic Slurm launcher. It requires the investigation agent to supply the implemented experiment entry point and verified environment setup. **It does not implement or launch the sensitivity analysis on its own.**

Open the workbook using the existing research-geo Jupyter kernel. Its code cells only inspect existing files and show the planned registry; the heavier read-only audits default to disabled. No cell submits Slurm jobs or runs graph construction automatically. The future agent should add scientific results only after completing the specified implementation and runs.

## How to delegate

Give the agent this instruction plus the prompt file:

> Execute the investigation in `docs/attachment_sensitivity/agent_prompt.md`, recording the work in `scripts/07_attachment_sensitivity_workbook.ipynb`. Run the full experiment locally if profiling supports it; otherwise finish the isolated implementation and tests and deliver a concrete Princeton Della submission package and execution instructions. Report only completed results and do not overwrite the production pipeline's outputs.

The prompt requires the agent to implement missing threshold/treatment/output options. Current production commands do not yet expose this full experiment. It also requires the same core statistical models, both demographic specifications, appropriate samples, and 199 successful bootstrap replications for final comparisons.

## Della handoff pattern -- finalize after implementation

Use [Princeton's Della page](https://researchcomputing.princeton.edu/systems/della) for current access and scheduler policy. CPU login is `della.princeton.edu`; Princeton VPN is required when off campus. Heavy work belongs in compute allocations. Check [Python environment guidance](https://researchcomputing.princeton.edu/support/knowledge-base/python), [memory requests](https://researchcomputing.princeton.edu/support/knowledge-base/memory), and [jobstats](https://researchcomputing.princeton.edu/support/knowledge-base/jobstats).

The commands below are a **submission pattern**, not a claim that remote paths/environments have been verified. The investigation agent must replace the paths with actual tested files and provide the missing implementation and data-staging commands. Repository data are git-ignored; cloning code alone is insufficient.

Connect from a terminal:

```bash
ssh YOUR_NETID@della.princeton.edu
```

On Della, after the agent has staged the experiment, checked input hashes, and provided working scripts:

```bash
export SLR_REPO='/verified/path/to/experiment-checkout'
export SLR_ENV_SETUP='/verified/path/to/environment_setup.sh'
export SLR_TASK_ENTRYPOINT='/verified/path/to/run_attachment_experiment.sh'
cd "$SLR_REPO"
mkdir -p logs

# First profile the actual full-network workload in a compute allocation.
# The template's 64G / 24-hour / 1-CPU request is provisional, not measured.
# Supply benchmark-supported --mem, --time, and --cpus-per-task overrides
# when necessary. The agent must provide those choices and their evidence.
SLR_SUBMISSION=$(sbatch --parsable --chdir="$SLR_REPO" \
  docs/attachment_sensitivity/della_job.sbatch.template)
SLR_JOB_ID=${SLR_SUBMISSION%%;*}
printf '%s\n' "$SLR_JOB_ID"
squeue -j "$SLR_JOB_ID"
tail -n 60 "logs/slr-attachment-${SLR_JOB_ID}.out"
sacct -j "$SLR_JOB_ID" --format=JobID,JobName,State,Elapsed,MaxRSS,AllocCPUS,ExitCode
jobstats "$SLR_JOB_ID"
```

To cancel only this investigation job if required:

```bash
scancel "$SLR_JOB_ID"
```

To retrieve the completed small report bundle, run locally after the agent supplies its actual remote location:

```bash
scp -r YOUR_NETID@della.princeton.edu:/verified/path/to/report_bundle ./attachment_sensitivity_report
```

The final agent handoff must also contain verified local commands, environment/package setup, exact input staging or links, per-stage logs, resume instructions, and any `afterok` dependency submissions. Do not regard queue submission, a smoke pass, or missing output files as a scientific result. Archive the final manifest, reports, code snapshot, and essential outputs somewhere durable; scratch is not a backup.

## Interpretation after completion

The planned contrast is not simply "all facilities versus some facilities." It separates tighter origin eligibility, tighter facility attachment, admitting previously excluded candidates, and removing the preference for two-edge-connected components. In the uncapped-addition arm, existing valid attachments remain exactly fixed. All candidates still come from the original spatial footprint.

If results are stable, state the maximum observed changes and the conclusions that persisted. If selected outcomes, locations, or associations change, report those qualifications. Neither stable results nor an unrestricted attachment rule establishes that every modeled connection is physically traversable.


### Generic guarded Slurm launcher

The agent must implement and test the referenced entry point and environment script. Resource values are provisional.

```bash
#!/bin/bash
# TEMPLATE ONLY: the investigation agent must create and test the entry point.
# Resource values below are provisional examples, NOT measured requirements.
# Override with sbatch --mem=... --time=... after profiling a full-network job.
#SBATCH --job-name=slr-attachment
#SBATCH --nodes=1
#SBATCH --ntasks=1
#SBATCH --cpus-per-task=1
#SBATCH --mem=64G
#SBATCH --time=24:00:00
#SBATCH --output=logs/slr-attachment-%j.out
#SBATCH --error=logs/slr-attachment-%j.err

set -euo pipefail

# Required absolute paths, set by the agent's completed submission instructions.
: "${SLR_REPO:?Set SLR_REPO to the verified experiment checkout}"
: "${SLR_ENV_SETUP:?Set SLR_ENV_SETUP to the verified environment setup script}"
: "${SLR_TASK_ENTRYPOINT:?Set SLR_TASK_ENTRYPOINT to the implemented experiment shell script}"
[[ -d "$SLR_REPO" ]] || { echo "Missing repository directory" >&2; exit 2; }
[[ -f "$SLR_ENV_SETUP" ]] || { echo "Missing environment setup script" >&2; exit 2; }
[[ -f "$SLR_TASK_ENTRYPOINT" ]] || { echo "Missing implemented entry point" >&2; exit 2; }
cd "$SLR_REPO"

# Some module/conda activation scripts require nounset to be disabled.
set +u
source "$SLR_ENV_SETUP"
set -u
export OMP_NUM_THREADS="${SLURM_CPUS_PER_TASK:-1}"
export OPENBLAS_NUM_THREADS="${SLURM_CPUS_PER_TASK:-1}"
export MKL_NUM_THREADS="${SLURM_CPUS_PER_TASK:-1}"
export PYTHONUNBUFFERED=1

# The entry point must use the tested Python/R interpreters, isolated outputs,
# explicit arm configurations, and its own validated completion markers.
srun bash "$SLR_TASK_ENTRYPOINT"

```

## 7. Results (experiment `attach_20260925`)

The cells below only read saved outputs; they start no computation. Full interpretation, deviations, and suggested manuscript text: `outputs/attachment_sensitivity/attach_20260925/REPORT.md`.

Execution record: scenarios 0-5 ft ran 2026-09-25/26; the machine shut down during the 6 ft union; the run resumed 2026-09-29 from atomically written parts (6 ft union via `coverage_union_all`, checked equal to `union_all` at 0 and 5 ft). Local only; peak RSS 7.35 GiB; no Slurm jobs.


In [ ]:
EXP = REPO / "outputs/attachment_sensitivity/attach_20260925"
pd.set_option("display.width", 200)
print("Validation:")
print(pd.read_csv(EXP / "validation/validation_summary.csv")[["check", "passed"]].to_string(index=False))
print("\nEligibility:")
print(pd.read_csv(EXP / "eligibility_comparison.csv").T.to_string())
print("\nFacility attachment changes:")
print(pd.read_csv(EXP / "attachment_audit_summary.csv")[["attachment_arm", "valid", "n_added", "n_excluded", "n_reattached", "reattach_displacement_m_p50", "reattach_displacement_m_max"]].to_string(index=False))
print("\nRanges across arms by scenario:")
print(pd.read_csv(EXP / "summary_ranges_by_scenario.csv").T.to_string())


In [ ]:
for spec in ["demographic_only", "with_physical"]:
    ame = pd.read_csv(EXP / f"ame_comparison_{spec}.csv")
    summary = ame.groupby("term").agg(
        positive=("estimate", lambda s: int((s > 0).sum())),
        significant_05=("p.value", lambda s: int((s < 0.05).sum())),
        min_pp=("estimate_pp", "min"), max_pp=("estimate_pp", "max"),
        max_abs_diff_vs_ref_pp=("abs_diff_pp", lambda s: s.abs().max()))
    print(f"\n{spec} (6 arms x 7 transitions = 42 AMEs per term)")
    print(summary.round(3).to_string())


## 8. Completion record

- Experiment implemented: **done** (`scripts/attachment_sensitivity/`), 2026-09-25
- Synthetic/equivalence tests: **11/11 pass** (`test_attachment_synthetic.py`); coverage-union equivalence check 2026-09-29
- Reference reproduced: **exact** -- all columns, 494,865 rows, facility and bridge audits, block-group dataset, and AME tables (<=4e-13)
- Full access classifications: **completed** for 6 arms x 7 scenarios, 2026-09-29 17:43 (local)
- Model fits and 199-success bootstrap tables: **completed** 2026-09-29 18:49 for 5 arms x 2 specs; facility_add_uncapped reused from reference (identical model input)
- Spatial attachment review: **GeoPackage produced**; geographic plausibility unverified except Fisher Island (ferry-only, general knowledge) and out-of-extent candidates
- Final Methods/Results wording: **suggested** in REPORT.md section 7; manuscript not edited
